# Straż Przyszłości: samodzielny launcher roju na Kaggle

Ten notatnik uruchamia istniejący kod selfopt_chain z repozytorium STRAZ_PRZYSZLOSCI. Nie kopiuje logiki roju: pobiera aktualny kod z GitHub, przechowuje stan w /kaggle/working, wykonuje ograniczoną serię cykli, a na końcu zapisuje archiwum stanu.

Architektura: basal (wbudowany lub opcjonalnie model basal-serve) -> orkiestrator -> OmniRoute (tylko jawnie zatwierdzone modele bezpłatne) -> wykonawca direct_patch albo izolowany Hermes -> weryfikacja -> propozycje do ręcznej recenzji. Bez automatycznego merge i bez zapisu do GitHub.

## Pierwszy start
1. Utwórz prywatny Kaggle Notebook Python z włączonym Internetem; zaimportuj ten plik .ipynb.
2. Uruchom wszystkie komórki z MODE = "demo". To jest test lokalnej pętli bez wywołań API.
3. Aby uruchamiać prawdziwe modele, ustaw MODE = "real", REAL_API_APPROVED = True i FREE_TIER_LIMITS_CONFIRMED = True. Najpierw aktywuj i zweryfikuj legalnie dostępne darmowe limity usług.
4. W Add-ons -> Secrets dodaj OMNIROUTE_ENDPOINT_KEY (aktywny klucz endpointu aktualnej instalacji OmniRoute), a przy wymaganym providerze np. NVIDIA_API_KEY. Nie wpisuj kluczy bezpośrednio w notatnik. Świeża instalacja OmniRoute nie tworzy automatycznie gotowych połączeń i może wymagać ich konfiguracji.
5. Jeśli masz prywatny, już skonfigurowany profil OmniRoute, możesz podać OMNIROUTE_CONFIG_INPUT wskazujący na jego katalog w dołączonym prywatnym Kaggle Dataset. Uwaga: taki katalog może zawierać poufne dane - nie publikuj datasetu ani outputów.
6. Opcjonalny prawdziwy basal-serve na CPU: BASAL_MODE = "mini_cpu". Domyślnie działa już wbudowany rdzeń bazujący na BasalDecisionCore.
7. Opcjonalny Hermes: HERMES_MODE = "isolated". Wymaga działającego bubblewrap/user namespaces. Jeżeli Kaggle blokuje izolację, notatnik przerywa tryb Hermes zamiast wykonywać obcy kod bez ochrony.

Kaggle nie jest usługą 24/7. Sesje mają ograniczony czas; zaplanuj uruchomienia w Kaggle Schedule. Archiwum swarm_state.zip należy zachować jako dane wejściowe następnej sesji (prywatny Dataset, ścieżka STATE_ARCHIVE_INPUT). Sam harmonogram nie przenosi stanu automatycznie.

Raporty i logi mogą zawierać treści zadań. Notebook oraz archiwa powinny pozostać prywatne.

In [ ]:
# KONFIGURACJA OPERATORA - jedyne miejsce wymagające edycji.
MODE = "demo"                      # "demo" lub "real"
REAL_API_APPROVED = False           # True = świadoma zgoda na wywołania sieciowe
FREE_TIER_LIMITS_CONFIRMED = False  # True dopiero po sprawdzeniu dostępności i limitów u dostawcy
ALLOW_NVIDIA_FREE_TIER = False      # True tylko przy potwierdzonym NVIDIA NIM free tier
BASAL_MODE = "builtin"              # "builtin" albo "mini_cpu"
HERMES_MODE = "off"                 # "off" albo "isolated" (eksperymentalne na Kaggle)
REPOSITORY = "StrazPrzyszlosci/STRAZ_PRZYSZLOSCI"
REPO_REF = "main"
OMNIROUTE_VERSION = "3.8.54"
MAX_CALLS_PER_DAY = 24
RPM_PER_MODEL = 4
RUN_SECONDS = 600
RUN_CYCLES = 2
MAX_JOBS = 1
EVOLUTION_TASKS = 1
STATE_ARCHIVE_INPUT = ""            # np. /kaggle/input/moj-stan/swarm_state.zip
OMNIROUTE_CONFIG_INPUT = ""         # opcjonalny prywatny katalog profilu ~/.omniroute
HERMES_RUNTIME_INPUT = ""           # opcjonalny gotowy katalog runtime z READY.json i bin/hermes
HERMES_BOOTSTRAP = False            # True: eksperymentalna instalacja ze źródeł NousResearch
import os, sys, json, time, shutil, zipfile, subprocess, pathlib, hashlib, secrets, urllib.request, urllib.error
from datetime import datetime, timezone
ROOT = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle").exists() else "/tmp/kaggle-swarm")
ROOT.mkdir(parents=True, exist_ok=True)
WORKTREE = ROOT / "stra z przyszlosci".replace(" ", "_")
STATE = ROOT / "swarm_state"
STATE.mkdir(exist_ok=True)
OUT = ROOT / "swarm_outputs"
OUT.mkdir(exist_ok=True)
def command(argv, *, cwd=None, env=None, timeout=600, log=None):
    completed = subprocess.run([str(x) for x in argv], cwd=cwd, env=env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    if log:
        pathlib.Path(log).write_text(completed.stdout, encoding="utf-8")
    if completed.returncode != 0:
        raise RuntimeError("Polecenie zakończone błędem: " + str(argv[0]) +
            ". Szczegóły w lokalnym logu; nie pokazuj logów zawierających sekrety.")
    return completed.stdout.strip()
def kaggle_secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        return ""
def get_json(url, key="", timeout=10):
    headers = {"Authorization": "Bearer " + key} if key else {}
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))
def wait_http(url, *, key="", seconds=50):
    until = time.monotonic() + seconds
    while time.monotonic() < until:
        try:
            get_json(url, key=key, timeout=3)
            return True
        except Exception:
            time.sleep(1)
    return False
assert MODE in ("demo", "real")
assert BASAL_MODE in ("builtin", "mini_cpu")
assert HERMES_MODE in ("off", "isolated")
assert RUN_SECONDS > 0 and RUN_SECONDS <= 36000 and 1 <= RUN_CYCLES <= 50
assert 1 <= MAX_CALLS_PER_DAY <= 1000 and 1 <= RPM_PER_MODEL <= 60
print("Tryb:", MODE, "| katalog stanu:", STATE)

In [ ]:
# Repozytorium i odtwarzanie stanu. Archiwum wczytywane tylko z własnego, zaufanego datasetu.
if STATE_ARCHIVE_INPUT:
    source = pathlib.Path(STATE_ARCHIVE_INPUT)
    if not source.is_file():
        raise FileNotFoundError("Nie znaleziono STATE_ARCHIVE_INPUT")
    with zipfile.ZipFile(source) as z:
        for entry in z.infolist():
            destination = (STATE / entry.filename).resolve()
            if not destination.is_relative_to(STATE.resolve()):
                raise RuntimeError("Niedozwolona ścieżka w archiwum")
            if entry.filename.startswith("/") or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError("Niedozwolony wpis archiwum")
        z.extractall(STATE)
if not (WORKTREE / ".git").is_dir():
    command(["git", "clone", "--depth", "1", "https://github.com/" + REPOSITORY + ".git", str(WORKTREE)], timeout=180)
if REPO_REF != "main":
    command(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=WORKTREE, timeout=180)
    command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=WORKTREE)
source_sha = command(["git", "rev-parse", "HEAD"], cwd=WORKTREE)
sys.path.insert(0, str(WORKTREE))
(OUT / "source.json").write_text(json.dumps({"repository": REPOSITORY, "ref": REPO_REF,
    "commit": source_sha, "mode": MODE, "created_utc": datetime.now(timezone.utc).isoformat()},
    indent=2), encoding="utf-8")
print("Pobrano repozytorium. Commit:", source_sha[:12])

In [ ]:
# Tryb demo używa wyłącznie silnika mock i wbudowanego BasalDecisionCore.
# Tryb realny: instalacja i uruchomienie OmniRoute na lokalnym porcie, wykrywanie zatwierdzonych modeli.
ENV = os.environ.copy()
ENV.update({"SELFOPT_NETWORK": "0", "SELFOPT_HERMES": "0"})
REGISTRY = STATE / "armed_registry.json"
SNAPSHOT = STATE / "quota_snapshot.json"
OMNIROUTE_LOG = OUT / "omniroute.log"
SERVICES = []
if MODE == "real":
    if not (REAL_API_APPROVED and FREE_TIER_LIMITS_CONFIRMED):
        raise RuntimeError("Real API wymaga świadomej zgody i potwierdzenia limitów bezpłatnych.")
    npm = shutil.which("npm")
    if not npm:
        raise RuntimeError("Brak npm. Wybierz obraz Kaggle z Node.js/npm i włącz Internet.")
    prefix = ROOT / "node_runtime"
    node_bin = prefix / "node_modules" / "node" / "bin"
    if not (node_bin / "node").exists():
        command([npm, "install", "--no-audit", "--no-fund", "--prefix", str(prefix), "node@24"], timeout=420, log=OUT/"node_install.log")
    ENV["PATH"] = str(node_bin) + os.pathsep + ENV.get("PATH", "")
    command([npm, "install", "--no-audit", "--no-fund", "--prefix", str(prefix),
        "omniroute@" + OMNIROUTE_VERSION], env=ENV, timeout=900, log=OUT/"omniroute_install.log")
    omnicli = prefix / "node_modules" / ".bin" / "omniroute"
    if not omnicli.exists():
        raise RuntimeError("Instalacja OmniRoute nie utworzyła CLI")
    ENV["DATA_DIR"] = str(ROOT / "omniroute_data")
    ENV["HOSTNAME"] = "127.0.0.1"
    ENV["PORT"] = "20128"
    ENV["STRICT_ZERO_COST"] = "1"
    ENV["CI"] = "1"
    ENV["INITIAL_PASSWORD"] = kaggle_secret("OMNIROUTE_ADMIN_PASSWORD") or secrets.token_urlsafe(28)
    ENV["JWT_SECRET"] = kaggle_secret("OMNIROUTE_JWT_SECRET") or secrets.token_urlsafe(36)
    ENV["API_KEY_SECRET"] = kaggle_secret("OMNIROUTE_DB_SECRET") or secrets.token_urlsafe(36)
    if OMNIROUTE_CONFIG_INPUT:
        src = pathlib.Path(OMNIROUTE_CONFIG_INPUT)
        if not src.is_dir():
            raise RuntimeError("OMNIROUTE_CONFIG_INPUT musi wskazywać istniejący prywatny katalog")
        shutil.copytree(src, pathlib.Path(ENV["DATA_DIR"]), dirs_exist_ok=True)
    for name in ("NVIDIA_API_KEY", "DEEPSEEK_API_KEY"):
        value = kaggle_secret(name)
        if value:
            ENV[name] = value
    with OMNIROUTE_LOG.open("w") as handle:
        service = subprocess.Popen([str(omnicli)], env=ENV, cwd=ROOT,
            stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
        SERVICES.append(service)
    if not wait_http("http://127.0.0.1:20128/v1/models", seconds=65):
        raise RuntimeError("OmniRoute nie wystartował lub wymaga konfiguracji. Sprawdź prywatnie omniroute.log.")
    endpoint_key = kaggle_secret("OMNIROUTE_ENDPOINT_KEY")
    try:
        catalog = get_json("http://127.0.0.1:20128/v1/models", endpoint_key)
    except Exception as error:
        raise RuntimeError("Endpoint OmniRoute odrzucił zapytanie. Skonfiguruj providera i klucz endpointu przed uruchomieniem realnym.") from error
    candidates = catalog.get("data", [])
    profiles = json.loads((WORKTREE / "selfopt_chain/runtime_model_profiles.json").read_text(encoding="utf-8"))["profiles"]
    selected = []
    for model in candidates:
        mid = model.get("id", "")
        if mid not in profiles:
            continue
        owned = model.get("owned_by")
        explicit = mid.startswith("kilocode/") and mid.endswith(":free") and owned == "kilocode"
        nvidia = ALLOW_NVIDIA_FREE_TIER and mid.startswith("nvidia/") and owned == "nvidia"
        if not (explicit or nvidia):
            continue
        context = model.get("context_length")
        if not isinstance(context, int) or context < 1024:
            continue
        profile = profiles[mid]
        selected.append({
          "model_id": mid, "provider": "nvidia_nim_free" if nvidia else "kilocode_free",
          "role": ",".join(profile["routing_roles"]), **profile,
          "privacy_class": "public_only", "cost_class": "free_quota",
          "context_tokens": context, "rpm_seed": RPM_PER_MODEL, "rpd_seed": MAX_CALLS_PER_DAY,
          "limits_verified": False,
          "supports_tools": model.get("capabilities", {}).get("tool_calling") is True if isinstance(model.get("capabilities"), dict) else False,
          "api": {"style": "openai_chat", "base_url": "http://127.0.0.1:20128/v1",
             "secret_env": "SELFOPT_OMNIROUTE_KEY",
             "response_model_id": mid.removeprefix("kilocode/") if explicit else mid,
             "max_output_tokens": model.get("max_output_tokens") or 4096}
        })
    if not selected:
        raise RuntimeError("Brak modeli z aktualnej allowlisty w katalogu OmniRoute. Nie wolno automatycznie wybierać płatnych.")
    REGISTRY.write_text(json.dumps({"models": selected}, indent=2), encoding="utf-8")
    from selfopt_chain.registry import load_registry
    load_registry(REGISTRY)
    now = datetime.now(timezone.utc).isoformat()
    SNAPSHOT.write_text(json.dumps({"generated_at": now, "entries": [
        {"provider": m["provider"], "model": m["model_id"], "source": "operator_verified_free_tier",
         "checked_at": now, "status": "ok", "rpm": RPM_PER_MODEL,
         "rpd": MAX_CALLS_PER_DAY, "tpm": None, "guaranteed": False,
         "note": "Operator confirmed free-tier eligibility; conservative local cap, upstream quota not guaranteed"}
        for m in selected]}, indent=2), encoding="utf-8")
    ENV.update({"SELFOPT_NETWORK": "1", "SELFOPT_OMNIROUTE_URL": "http://127.0.0.1:20128/v1",
         "SELFOPT_OMNIROUTE_KEY": endpoint_key})
    print("OmniRoute: aktywne, zatwierdzone modele:", [m["model_id"] for m in selected])
else:
    print("Demo: OmniRoute nie jest uruchamiany, nie wykonujemy połączeń z płatnymi API.")

In [ ]:
# Opcjonalne uruchomienie realnego modelu basal oraz izolowanego Hermes.
if MODE == "real" and BASAL_MODE == "mini_cpu":
    command([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "basal @ https://github.com/rkinas/basal/archive/refs/tags/v2.1.0.tar.gz"],
        env=ENV, timeout=900, log=OUT/"basal_install.log")
    with (OUT/"basal.log").open("w") as handle:
        process = subprocess.Popen(["basal-serve", "--model", "Remek/basal-2.0-mini",
            "--mode", "eager", "--device", "cpu", "--port", "8000"],
            env=ENV, cwd=ROOT, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
        SERVICES.append(process)
    # /health może nie być JSON: sprawdzamy sam status HTTP.
    healthy = False
    until = time.monotonic() + 240
    while time.monotonic() < until:
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3) as response:
                healthy = response.status == 200
            if healthy:
                break
        except Exception:
            time.sleep(2)
    if not healthy:
        raise RuntimeError("basal-serve nie osiągnął gotowości; sprawdź basal.log.")
    ENV["SELFOPT_BASAL_URL"] = "http://127.0.0.1:8000"
if MODE == "real" and HERMES_MODE == "isolated":
    if not shutil.which("bwrap"):
        raise RuntimeError("Hermes wymaga bwrap. Kaggle nie zapewnia tu izolacji; nie uruchamiam workera.")
    probe = subprocess.run(["bwrap", "--unshare-all", "--ro-bind", "/usr", "/usr",
       "--ro-bind", "/bin", "/bin", "--dev", "/dev", "--proc", "/proc", "--", "/usr/bin/true"],
       capture_output=True, timeout=15)
    if probe.returncode != 0:
        raise RuntimeError("Kaggle zablokował bubblewrap/user namespaces; Hermes nie może pracować bezpiecznie.")
    if HERMES_RUNTIME_INPUT:
        runtime = pathlib.Path(HERMES_RUNTIME_INPUT)
    elif HERMES_BOOTSTRAP:
        # Przygotowanie izolowanego runtime wg tools/prepare_hermes_runtime.py.
        # Instalacja ze źródeł jest eksperymentalna; nie pobiera kluczy.
        src = ROOT/"hermes_source"
        if not (src/".git").exists():
            command(["git", "clone", "--depth", "1", "https://github.com/NousResearch/hermes-agent.git", str(src)], timeout=200)
        command([sys.executable, "-m", "pip", "install", "uv"], timeout=240, log=OUT/"uv_install.log")
        command(["uv", "venv", "--python", "3.12", str(src/".venv")], timeout=240)
        command(["uv", "pip", "install", "--python", str(src/".venv/bin/python"), "-e", str(src)], timeout=900, log=OUT/"hermes_install.log")
        built = command([sys.executable, str(WORKTREE/"tools/prepare_hermes_runtime.py"),
            "--source", str(src), "--destination", str(ROOT/"hermes_runtime")], timeout=600)
        runtime = pathlib.Path(built.splitlines()[-1])
    else:
        raise RuntimeError("Podaj HERMES_RUNTIME_INPUT lub ustaw HERMES_BOOTSTRAP=True; brak gotowego izolowanego Hermes.")
    if not (runtime/"READY.json").is_file() or not (runtime/"bin/hermes").is_file():
        raise RuntimeError("Hermes runtime niekompletny (READY.json lub bin/hermes)")
    ENV["SELFOPT_HERMES"] = "1"
    ENV["SELFOPT_HERMES_RUNTIME"] = str(runtime)
print("Decyzje:", "basal-serve" if "SELFOPT_BASAL_URL" in ENV else "BasalDecisionCore wbudowany",
      "| Hermes:", "izolowany" if ENV.get("SELFOPT_HERMES") == "1" else "wyłączony")

In [ ]:
# Uruchomienie istniejącego orkiestratora i bezpieczny eksport stanu.
import signal
args = [sys.executable, "selfopt_chain/orchestrator.py",
    "--engine", "omniroute" if MODE == "real" else "mock",
    "--state-dir", str(STATE),
    "--kill-switch", str(STATE/"KILL_SWITCH"),
    "--quota-ledger", str(STATE/"quota_ledger.json"),
    "--max-seconds", str(RUN_SECONDS),
    "--max-cycles", str(RUN_CYCLES),
    "--max-calls", str(MAX_CALLS_PER_DAY),
    "--max-jobs", str(MAX_JOBS),
    "--evolve-max-tasks", str(EVOLUTION_TASKS),
    "--hermes-max-turns", "6",
    "--hermes-run-seconds", "180",
    "--interval", "10",
    "--evolve", "--evolve-executor", "hermes" if ENV.get("SELFOPT_HERMES") == "1" else "direct_patch"]
if MODE == "real":
    args += ["--registry", str(REGISTRY), "--quota-snapshot", str(SNAPSHOT)]
else:
    args += ["--once"]
if (STATE/"KILL_SWITCH").exists():
    raise RuntimeError("W katalogu stanu jest KILL_SWITCH. Usuń go świadomie przed nowym uruchomieniem.")
exit_code = None
run_log = OUT/"orchestrator.log"
try:
    with run_log.open("w", encoding="utf-8") as stream:
        process = subprocess.Popen(args, cwd=WORKTREE, env=ENV, stdout=stream,
            stderr=subprocess.STDOUT, start_new_session=True)
        try:
            exit_code = process.wait(timeout=RUN_SECONDS+45)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=8)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            exit_code = 124
finally:
    for service in SERVICES:
        if service.poll() is None:
            try:
                os.killpg(service.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    output_archive = OUT/"swarm_state.zip"
    # Stan nie zawiera konfiguracji ani kluczy serwera OmniRoute.
    exclude_suffixes = (".pem", ".key", ".env", ".sqlite", ".db")
    with zipfile.ZipFile(output_archive, "w", zipfile.ZIP_DEFLATED) as dest:
        for file in sorted(STATE.rglob("*")):
            if file.is_file() and not file.is_symlink() and not file.name.startswith(".") \
                and not file.name.endswith(exclude_suffixes):
                dest.write(file, file.relative_to(STATE))
    summary = {"mode": MODE, "exit_code": exit_code, "repo_commit": source_sha,
        "finished_utc": datetime.now(timezone.utc).isoformat(),
        "archive": str(output_archive), "state_files": sum(1 for f in STATE.rglob("*") if f.is_file()),
        "hermes_activated": ENV.get("SELFOPT_HERMES") == "1",
        "basal_remote": ENV.get("SELFOPT_BASAL_URL") is not None,
        "manual_review_required": True, "auto_merged": False}
    (OUT/"run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Dostępne outputy:", [str(x) for x in sorted(OUT.iterdir()) if x.suffix in (".json", ".zip")])
if exit_code != 0:
    print("Uwaga: proces zgłosił błąd lub stan degraded. Sprawdź prywatny log orchestrator.log i artifacts/receipt.")

## Po uruchomieniu

Pobierz z Output pliki swarm_state.zip i run_summary.json. Archiwum zawiera ledger, receipty i propozycje z kwarantanny, ale nie zawiera konfiguracji OmniRoute. Aby kontynuować poprzedni stan, umieść archiwum w prywatnym Kaggle Dataset i ustaw STATE_ARCHIVE_INPUT w pierwszej komórce.

Brak GPU nie przeszkadza w trybie demo i wbudowanym Basal. Prawdziwy basal-serve na CPU jest opcjonalny i może być powolny. Kaggle nie gwarantuje obsługi bubblewrap. Propozycje roju nie są automatycznie scalane ani publikowane.

Ważne: włączenie MODE = "real" nie jest dowodem dostępności darmowych modeli; wymagana jest aktywna konfiguracja providerów, endpoint OmniRoute oraz bieżąca weryfikacja limitów. Jeśli tych elementów brakuje, notebook ma zakończyć się błędem, nie używać płatnego fallbacku.